In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
PROJECT_DIR = Path("/content/project")
BRANCH = __import__("os").environ.get("VTC_GIT_REF", "main")

# Public GitHub repository: no GITHUB_TOKEN or Colab Secret needed.
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"],
        capture_output=True, text=True, check=True,
    )
    if dirty.stdout.strip():
        raise RuntimeError("Local /content/project contains uncommitted changes; resolve them first")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"],
        check=True,
    )
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f"{PROJECT_DIR} exists and is not a Git repository")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

print("Repository ready:", PROJECT_DIR)


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]"],
    cwd=PROJECT_DIR, check=True,
)
SRC_DIR = (PROJECT_DIR / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from vietnamese_topic_classification.config import load_domain_config
from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake

config = load_domain_config()

print("Task:", config["task"])
print("Classes:", config["num_classes"])
print("Dataset:", config["dataset"]["id"])
print("Revision:", config["dataset"]["revision"])

In [ ]:
from google.colab import drive
from pathlib import Path

MOUNT = "/content/vtsnlp_drive"

drive.mount(MOUNT)

# Each teammate mounts their own Drive account. Configure the shortcut
# or DATA_LAKE_ROOT environment variable for shared Drive access.
DRIVE_ROOT = resolve_mounted_lake(MOUNT)

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy project folder: {DRIVE_ROOT}"
    )

print("Google Drive connected!")
print("Project:", DRIVE_ROOT)

In [ ]:
import subprocess
import sys

SAMPLE_SIZE = 50_000
SEED = 42
SHUFFLE_BUFFER = 10_000

revision = config["dataset"]["revision"]

STAMP = (
    f"{revision[:10]}_seed{SEED}"
    f"_buf{SHUFFLE_BUFFER}_n{SAMPLE_SIZE}"
)

BRONZE_FILE = (
    DRIVE_ROOT / "01_bronze"
    / f"vtsnlp_raw_sample_{STAMP}.jsonl"
)

MANIFEST_FILE = (
    DRIVE_ROOT / "00_source_metadata"
    / f"source_manifest_{STAMP}.json"
)

REPORT_FILE = (
    DRIVE_ROOT / "04_reports"
    / f"bronze_ingestion_{STAMP}.json"
)

from vietnamese_topic_classification.data.bronze import BronzeReader

BRONZE_ALREADY_EXISTS = False
existing = [p for p in (BRONZE_FILE, MANIFEST_FILE, REPORT_FILE) if p.exists()]
if existing:
    if len(existing) != 3:
        raise FileExistsError(f"Partial Bronze snapshot found: {existing}")
    existing_reader = BronzeReader(DRIVE_ROOT, MANIFEST_FILE.name)
    verified = existing_reader.verify()
    if verified["records"] != SAMPLE_SIZE:
        raise AssertionError("Existing Bronze row count differs from the configuration")
    BRONZE_ALREADY_EXISTS = True
    print("EXISTING BRONZE VERIFIED — will skip ingestion:", verified)
else:
    print("No Bronze snapshot yet. Ingestion will create a new immutable snapshot.")

command = [
    sys.executable,
    str(PROJECT_DIR / "scripts/01_ingest_bronze.py"),
    "--output-root", str(DRIVE_ROOT),
    "--sample-size", str(SAMPLE_SIZE),
    "--seed", str(SEED),
    "--shuffle-buffer", str(SHUFFLE_BUFFER),
]
if BRONZE_ALREADY_EXISTS:
    print("BRONZE INGESTION SKIPPED — snapshot already verified")
else:
    subprocess.run(command, cwd=str(PROJECT_DIR), check=True)
    print("BRONZE INGESTION COMPLETED")

In [ ]:
import hashlib
import json

assert BRONZE_FILE.is_file(), "Missing Bronze file"
assert MANIFEST_FILE.is_file(), "Missing manifest"
assert REPORT_FILE.is_file(), "Missing report"

manifest = json.loads(
    MANIFEST_FILE.read_text(encoding="utf-8")
)

report = json.loads(
    REPORT_FILE.read_text(encoding="utf-8")
)

digest = hashlib.sha256()
rows = 0

with BRONZE_FILE.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(chunk)
        rows += chunk.count(b"\n")

assert rows == manifest["sample_size"]
assert digest.hexdigest() == manifest["sha256"]

print("BRONZE VALIDATION PASSED")
print("Records:", rows)
print("File size:", BRONZE_FILE.stat().st_size)
print("SHA256:", digest.hexdigest())
print("Domains:", len(report["domain_counts"]))
print("Source:", manifest["dataset"])
print("Revision:", manifest["revision"])